## **Estimación local de coeficientes aerodinámicos en una pila circular mediante probes de pared en LES‑WALE**
 
Se presenta una metodología para estimar coeficientes locales de sustentación, arrastre y momento a partir de datos de presión y velocidad obtenidos con probes en una simulación LES‑WALE de un cuerpo cilíndrico. El procedimiento abarca:

- Definición de sondas en el archivo `controlDict`
- Extracción de series temporales de $p$, $U$ y $nut$
- Reconstrucción de esfuerzos tangenciales con una aproximación local de gradiente
- Cálculo de coeficientes aerodinámicos locales y sus estadísticas temporales.<br>

La metodología permite estudiar la variación azimutal de fuerzas y su comportamiento fluctuante con un costo computacional reducido.

### **1. Introducción**  
En flujos alrededor de pilas circulares, la distribución espacial de fuerzas locales es esencial para analizar fluctuaciones asociadas al desprendimiento de vórtices, los mecanismos de transferencia de energía y la dinámica de cargas estructurales. Aunque OpenFOAM proporciona funciones integrales como `forceCoeffsIncompressible`, estas devuelven fuerzas globales sobre parches y no permiten evaluar variaciones locales ni asociarlas directamente a series temporales de probes. Se propone aquí un enfoque basado en sondas ubicadas sobre la pared (o a distancia controlada), que permite reconstruir localmente presión y esfuerzo cortante a partir de campos de LES‑WALE.

### **2. Definición de probes en `controlDict`**  
Se recomienda definir una familia de probes distribuidas azimutalmente alrededor del cilindro, ubicadas sobre la pared o a una distancia conocida y. Los campos mínimos requeridos son `p`, `U` y `nut`. Un ejemplo simplificado de definición es:

```foam
functions
{
    wallProbes
    {
        type            probes;
        libs            ("libsampling.so");
        writeControl    timeStep;
        writeInterval   1;
        fields          (p U nut);

        probeLocations
        (
            (x1 y1 z1)
            (x2 y2 z2)
            ...
        );
    }
}
```

Las coordenadas deben corresponder a puntos en la superficie cilíndrica. Si se utiliza una distancia fija al muro, debe registrarse con precisión para evaluar el gradiente.

### **3. Reconstrucción de magnitudes locales**  
Sea un cilindro de radio $R$ con centro $(x_0,y_0,z_0)$ y eje $z$. Para cada probe en $\mathbf{r}=(x,y,z)$, el ángulo azimutal se define como:

$$\theta=\arctan2(y-y_0,\;x-x_0)$$

La normal de pared (radial) es:

$$\mathbf{n}=\frac{1}{R}(x-x_0,\;y-y_0,\;0)$$

En LES‑WALE, la viscosidad efectiva es:

$$\nu_{\mathrm{eff}}=\nu+\nu_t,\qquad \mu_{\mathrm{eff}}=\rho\,\nu_{\mathrm{eff}}$$

La velocidad se descompone en componentes normal y tangencial:

$$\mathbf{U}_n=(\mathbf{U}\cdot\mathbf{n})\mathbf{n},\qquad
\mathbf{U}_t=\mathbf{U}-\mathbf{U}_n$$

Suponiendo un perfil lineal entre pared y probe a distancia $y$, el esfuerzo cortante local se aproxima por:

$$\tau_w \approx \mu_{\mathrm{eff}}\frac{|\mathbf{U}_t|}{y}$$

y su dirección tangencial es:

$$\hat{\mathbf{t}}=\frac{\mathbf{U}_t}{|\mathbf{U}_t|+\epsilon},\qquad
\boldsymbol{\tau}_w=\tau_w\,\hat{\mathbf{t}}$$

### **4. Fuerza local por unidad de área**  
La fuerza local por unidad de área sobre la superficie se define como:

$$
\mathbf{f}(\theta)=-p\,\mathbf{n}+\boldsymbol{\tau}_w
$$

La proyección sobre direcciones de interés (arrastre y sustentación) se obtiene con:

$$
f_D(\theta)=\mathbf{f}\cdot\mathbf{d},\qquad
f_L(\theta)=\mathbf{f}\cdot\mathbf{l}
$$

donde $\mathbf{d}$ y $\mathbf{l}$ son las direcciones unitarias de arrastre y sustentación, respectivamente.

### **5. Coeficientes locales**  
Los coeficientes hidrodinámicos locales por unidad de área se calculan con:

$$C_D(\theta)=\frac{f_D(\theta)}{\frac12\rho U_\infty^2},\qquad
C_L(\theta)=\frac{f_L(\theta)}{\frac12\rho U_\infty^2}$$

Para el momento respecto a un centro de referencia $\mathbf{r}_{CofR}$ y un eje $\mathbf{a}$:

$$\mathbf{M}=(\mathbf{r}-\mathbf{r}_{CofR})\times\mathbf{f},\qquad
M_{\mathrm{pitch}}=\mathbf{M}\cdot\mathbf{a}$$

$$C_M(\theta)=\frac{M_{\mathrm{pitch}}}{\frac12\rho U_\infty^2 L_{\mathrm{ref}}}$$

### **6. Estadísticas temporales: medias y fluctuantes**  
A partir de la serie temporal $C(t,\theta)$:

$$
\overline{C}(\theta)=\frac{1}{T}\int_0^T C(t,\theta)\,dt
$$

$$
C'(t,\theta)=C(t,\theta)-\overline{C}(\theta)
$$

$$
C_{\mathrm{rms}}(\theta)=\sqrt{\overline{C'(t,\theta)^2}}
$$

Estas métricas permiten evaluar tanto la estructura media como la intensidad de fluctuación en función del ángulo azimutal.